# Fatigue and Speed Analysis

This notebook combines game extraction, play-by-play visualization, velocity analysis, and spacing analysis to study player fatigue and speed patterns in basketball games.

## Overview

We will analyze:
- Player and team velocities throughout games
- Fatigue patterns across quarters
- Team spacing using convex hull analysis
- Correlations between speed, spacing, and fatigue

## Setup and Imports

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sys
import os

# Add game folder to path
sys.path.insert(0, os.path.join(os.getcwd(), 'game'))

from game import Game
import python_files.space as sp
from python_files.data_extracter import json_extracter

%matplotlib inline

## Game Data Extraction

We'll use the `extract_games()` function to load available games from the data directory.

In [ ]:
def extract_games():
    """
    Extract games from allgames.txt
    
    Returns:
        list: list of games. Each element is tuple (date, home_team, away_team)
        example element: ('01.01.2016', 'TOR', 'CHI')
    """
    with open('game/allgames.txt', 'r') as f:
        games = []
        for line in f:
            # Parse game filename: MMDDYYYY.AWAYTEAM.HOMETEAM.7z
            parts = line.strip().replace('.7z', '').split('.')
            if len(parts) == 3:
                date_str = parts[0]
                # Convert MMDDYYYY to MM.DD.YYYY
                date = f"{date_str[0:2]}.{date_str[2:4]}.{date_str[4:8]}"
                away_team = parts[1]
                home_team = parts[2]
                games.append((date, home_team, away_team))
    return games

# Extract available games
available_games = extract_games()
print(f"Found {len(available_games)} games")
print("\nFirst 5 games:")
for game in available_games[:5]:
    print(f"  {game[0]}: {game[2]} vs {game[1]}")

FileNotFoundError: [Errno 2] No such file or directory: 'allgames.txt'

## Load Sample Game Data

Let's load a sample game to demonstrate the analysis capabilities.

In [ ]:
# Load a sample game
data, events = json_extracter('data/game1.json')
print(f"Loaded game with {len(events)} events")

## Play-by-Play Visualization

Using the Game class, we can visualize individual plays and track player movements.

In [ ]:
# Create a Game object for visualization
# Note: This requires the game data in the proper format
# game = Game('path/to/game/data')
# game.watch_play(event_index=0)

print("Game visualization methods available:")
print("  - watch_play(event_index): Display a specific play")
print("  - animate_play(event_index): Animate player movements")

## Velocity Analysis

Calculate and analyze player velocities throughout the game.

In [ ]:
def calculate_velocities(events, event_id):
    """
    Calculate velocities for all players in an event
    
    Args:
        events: List of game events
        event_id: Index of the event to analyze
    
    Returns:
        dict: Player velocities
    """
    event = events[event_id]
    moments = event['moments']
    
    velocities = []
    for i in range(len(moments) - 1):
        moment1 = moments[i]
        moment2 = moments[i + 1]
        
        if sp.test_moment(moment1) and sp.test_moment(moment2):
            mom_infos = sp.players_ball_speed_position(moment1, moment2)
            velocities.append(mom_infos)
    
    return velocities

# Example: Calculate velocities for first event
if len(events) > 0:
    velocities = calculate_velocities(events, 0)
    print(f"Calculated velocities for {len(velocities)} moments")

### Velocity Statistics

Generate statistics about player and team velocities.

In [ ]:
def get_velocity_stats(velocities):
    """
    Calculate velocity statistics
    
    Args:
        velocities: List of velocity data from calculate_velocities
    
    Returns:
        dict: Statistics including mean, max, min velocities
    """
    team1_speeds = []
    team2_speeds = []
    
    for v in velocities:
        for player_id in v['team1']:
            speed = np.linalg.norm(v['team1'][player_id]['v'])
            team1_speeds.append(speed)
        
        for player_id in v['team2']:
            speed = np.linalg.norm(v['team2'][player_id]['v'])
            team2_speeds.append(speed)
    
    return {
        'team1': {
            'mean': np.mean(team1_speeds) if team1_speeds else 0,
            'max': np.max(team1_speeds) if team1_speeds else 0,
            'min': np.min(team1_speeds) if team1_speeds else 0
        },
        'team2': {
            'mean': np.mean(team2_speeds) if team2_speeds else 0,
            'max': np.max(team2_speeds) if team2_speeds else 0,
            'min': np.min(team2_speeds) if team2_speeds else 0
        }
    }

# Example usage
if len(events) > 0:
    stats = get_velocity_stats(velocities)
    print("\nVelocity Statistics:")
    print(f"Team 1 - Mean: {stats['team1']['mean']:.2f} ft/s, Max: {stats['team1']['max']:.2f} ft/s")
    print(f"Team 2 - Mean: {stats['team2']['mean']:.2f} ft/s, Max: {stats['team2']['max']:.2f} ft/s")

## Spacing Analysis

Analyze team spacing using convex hull calculations.

In [ ]:
from scipy.spatial import ConvexHull

def calculate_spacing(events, event_id, moment_id):
    """
    Calculate team spacing using convex hull
    
    Args:
        events: List of game events
        event_id: Index of the event
        moment_id: Index of the moment within the event
    
    Returns:
        dict: Convex hull areas for each team
    """
    event = events[event_id]
    moment = event['moments'][moment_id]
    
    # Extract player positions
    players = moment[5]
    team1_positions = []
    team2_positions = []
    
    for player in players[1:]:  # Skip ball
        team_id = player[0]
        x, y = player[2], player[3]
        
        if team_id == players[1][0]:  # First team
            team1_positions.append([x, y])
        else:
            team2_positions.append([x, y])
    
    # Calculate convex hulls
    team1_hull = ConvexHull(team1_positions) if len(team1_positions) >= 3 else None
    team2_hull = ConvexHull(team2_positions) if len(team2_positions) >= 3 else None
    
    return {
        'team1_area': team1_hull.volume if team1_hull else 0,
        'team2_area': team2_hull.volume if team2_hull else 0
    }

# Example usage
if len(events) > 0 and len(events[0]['moments']) > 0:
    spacing = calculate_spacing(events, 0, 0)
    print(f"\nTeam Spacing (Convex Hull Area):")
    print(f"Team 1: {spacing['team1_area']:.2f} sq ft")
    print(f"Team 2: {spacing['team2_area']:.2f} sq ft")

## Combined Analysis

Analyze the relationship between velocity, spacing, and fatigue.

In [ ]:
def analyze_fatigue_patterns(events):
    """
    Analyze fatigue patterns across quarters
    
    Args:
        events: List of game events
    
    Returns:
        dict: Fatigue metrics by quarter
    """
    # This is a placeholder for more sophisticated fatigue analysis
    # In practice, you would analyze velocity decline over time,
    # spacing changes, and other indicators of fatigue
    
    print("Fatigue analysis would include:")
    print("  - Velocity trends across quarters")
    print("  - Spacing changes over time")
    print("  - Player-specific fatigue indicators")
    print("  - Team-level fatigue patterns")
    
    return {}

analyze_fatigue_patterns(events)

## Visualization

Create visualizations to explore the relationships between different metrics.

In [ ]:
# Placeholder for visualization code
# This would include plots of:
# - Velocity over time
# - Spacing over time
# - Correlations between metrics

print("Visualization capabilities:")
print("  - Velocity time series")
print("  - Spacing evolution")
print("  - Fatigue indicators")
print("  - Correlation plots")